In [1]:
import gzip
import json
import ast
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

In [7]:
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre"

CLEAN_BOOKS_DIR = "/content/data"

GENRES = {
    "children": {
        "interactions_file": "goodreads_interactions_children.json.gz",
        "books_path": f"{CLEAN_BOOKS_DIR}/children.csv",
    },
    "comics": {
        "interactions_file": "goodreads_interactions_comics_graphic.json.gz",
        "books_path": f"{CLEAN_BOOKS_DIR}/comics.csv",
    },
    "fantasy": {
        "interactions_file": "goodreads_interactions_fantasy_paranormal.json.gz",
        "books_path": f"{CLEAN_BOOKS_DIR}/fantasy.csv",
    },
    "history": {
        "interactions_file": "goodreads_interactions_history_biography.json.gz",
        "books_path": f"{CLEAN_BOOKS_DIR}/history.csv",
    },
    "mystery": {
        "interactions_file": "goodreads_interactions_mystery_thriller_crime.json.gz",
        "books_path": f"{CLEAN_BOOKS_DIR}/mystery.csv",
    },
    "poetry": {
        "interactions_file": "goodreads_interactions_poetry.json.gz",
        "books_path": f"{CLEAN_BOOKS_DIR}/poetry.csv",
    },
    "romance": {
        "interactions_file": "goodreads_interactions_romance.json.gz",
        "books_path": f"{CLEAN_BOOKS_DIR}/romance.csv",
    },
    "young": {
        "interactions_file": "goodreads_interactions_young_adult.json.gz",
        "books_path": f"{CLEAN_BOOKS_DIR}/young.csv",
    },
}

In [8]:
def parse_list_col(x):
    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []

    return []

In [9]:
def download_file(url, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists():
        print(f"Already exists: {output_path}")
        return output_path

    cmd = f'wget -c -O "{output_path}" "{url}"'
    print(cmd)
    !{cmd}

    return output_path

In [10]:
def filter_read_interactions(input_path, output_dir, chunk_size=500_000):
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    buffer = []
    part = 0

    with gzip.open(input_path, "rt", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Filtering {input_path.name}"):
            buffer.append(json.loads(line))

            if len(buffer) >= chunk_size:
                chunk = pd.DataFrame(buffer)

                chunk["is_read"] = chunk["is_read"].astype(str).str.lower().eq("true")
                chunk["rating"] = pd.to_numeric(chunk["rating"], errors="coerce")

                chunk = chunk[chunk["is_read"]].copy()
                chunk = chunk[["user_id", "book_id", "rating"]].copy()

                if len(chunk) > 0:
                    chunk.to_parquet(output_dir / f"part_{part:04d}.parquet", index=False)
                    part += 1

                buffer = []

    if buffer:
        chunk = pd.DataFrame(buffer)

        chunk["is_read"] = chunk["is_read"].astype(str).str.lower().eq("true")
        chunk["rating"] = pd.to_numeric(chunk["rating"], errors="coerce")

        chunk = chunk[chunk["is_read"]].copy()
        chunk = chunk[["user_id", "book_id", "rating"]].copy()

        if len(chunk) > 0:
            chunk.to_parquet(output_dir / f"part_{part:04d}.parquet", index=False)

    return output_dir

In [11]:
def map_book_to_work(interactions_parquet_dir, books_path, output_path):
    interactions_parquet_dir = Path(interactions_parquet_dir)
    books_path = Path(books_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    df = pd.read_parquet(interactions_parquet_dir)

    df["book_id"] = df["book_id"].astype(str).str.strip()
    df["user_id"] = df["user_id"].astype(str).str.strip()
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    df = df.dropna(subset=["user_id", "book_id", "rating"]).copy()

    books_df = pd.read_csv(books_path)

    books_df["book_id_list"] = books_df["book_id_list"].apply(parse_list_col)

    book_to_work = (
        books_df[["work_id", "book_id_list"]]
        .explode("book_id_list")
        .rename(columns={"book_id_list": "book_id"})
        .dropna(subset=["book_id"])
        .copy()
    )

    book_to_work["book_id"] = book_to_work["book_id"].astype(str).str.strip()
    book_to_work["work_id"] = book_to_work["work_id"].astype(str).str.strip()
    book_to_work = book_to_work.drop_duplicates(subset=["book_id"])

    interaction_book_ids = set(df["book_id"].unique())
    mapping_book_ids = set(book_to_work["book_id"].unique())
    overlap = interaction_book_ids & mapping_book_ids

    print("unique interaction book_ids:", len(interaction_book_ids))
    print("unique mapped book_ids:", len(mapping_book_ids))
    print("overlap:", len(overlap))
    print("overlap rate:", len(overlap) / len(interaction_book_ids))

    df_work = df.merge(book_to_work, on="book_id", how="inner")

    print("before merge:", len(df))
    print("after merge:", len(df_work))
    print("coverage:", len(df_work) / len(df) if len(df) else 0)

    df_work = df_work[["user_id", "work_id", "rating"]].copy()

    df_work = (
        df_work
        .sort_values(["user_id", "work_id", "rating"], ascending=[True, True, False])
        .drop_duplicates(subset=["user_id", "work_id"], keep="first")
        .copy()
    )

    df_work.to_parquet(output_path, index=False)

    return df_work

In [12]:
from pathlib import Path

RAW_DIR = Path("/content/data/interactions")
READ_PARTS_DIR = Path("/content/data/processed/read_interaction_parts")
WORK_MAPPED_DIR = Path("/content/data/processed/work_mapped_interactions")

RAW_DIR.mkdir(parents=True, exist_ok=True)
READ_PARTS_DIR.mkdir(parents=True, exist_ok=True)
WORK_MAPPED_DIR.mkdir(parents=True, exist_ok=True)

for genre, cfg in GENRES.items():
    print("=" * 80)
    print(f"Processing: {genre}")

    interactions_file = cfg["interactions_file"]
    books_path = cfg["books_path"]

    # interactions are downloaded from UCSD
    url = f"{BASE_URL}/{interactions_file}"
    raw_path = RAW_DIR / interactions_file

    # output paths
    read_parts_out = READ_PARTS_DIR / genre
    work_mapped_out = WORK_MAPPED_DIR / f"{genre}_read_work_mapped.parquet"

    # 1. Download raw interactions json.gz
    download_file(url, raw_path)

    # 2. Filter is_read=True and save parquet parts
    filter_read_interactions(
        input_path=raw_path,
        output_dir=read_parts_out,
        chunk_size=500_000
    )

    # 3. Map book_id -> work_id using local cleaned books CSV
    df_work = map_book_to_work(
        interactions_parquet_dir=read_parts_out,
        books_path=books_path,
        output_path=work_mapped_out
    )

    print(f"Saved: {work_mapped_out}")
    print("Final shape:", df_work.shape)

Processing: children
wget -c -O "/content/data/interactions/goodreads_interactions_children.json.gz" "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_children.json.gz"
--2026-06-04 04:30:02--  https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_children.json.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 511288540 (488M) [application/gzip]
Saving to: ‘/content/data/interactions/goodreads_interactions_children.json.gz’

/content/data/inter 100%[===================>] 487.60M  5.80MB/s    in 65s     

2026-06-04 04:31:09 (7.49 MB/s) - ‘/content/data/interactions/goodreads_interactions_children.json.gz’ saved [511288540/511288540]



Filtering goodreads_interactions_children.json.gz: 0it [00:00, ?it/s]

unique interaction book_ids: 123196
unique mapped book_ids: 82394
overlap: 81833
overlap rate: 0.6642504626773597
before merge: 6626989
after merge: 5352015
coverage: 0.8076088552433088
Saved: /content/data/processed/work_mapped_interactions/children_read_work_mapped.parquet
Final shape: (5340525, 3)
Processing: comics
wget -c -O "/content/data/interactions/goodreads_interactions_comics_graphic.json.gz" "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_comics_graphic.json.gz"
--2026-06-04 04:33:52--  https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_comics_graphic.json.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 387419261 (369M) [application/gzip]
Saving to: ‘/content/data/interactions/goodreads_interactions_comics_graphic.json.gz’



Filtering goodreads_interactions_comics_graphic.json.gz: 0it [00:00, ?it/s]

unique interaction book_ids: 89278
unique mapped book_ids: 40179
overlap: 40120
overlap rate: 0.4493828266762248
before merge: 4764133
after merge: 3431607
coverage: 0.7203004198245515
Saved: /content/data/processed/work_mapped_interactions/comics_read_work_mapped.parquet
Final shape: (3427126, 3)
Processing: fantasy
wget -c -O "/content/data/interactions/goodreads_interactions_fantasy_paranormal.json.gz" "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_fantasy_paranormal.json.gz"
--2026-06-04 04:36:48--  https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_fantasy_paranormal.json.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2800816612 (2.6G) [application/gzip]
Saving to: ‘/content/data/interactions/goodreads_interactions_fantasy_parano

...

unique interaction book_ids: 36373
unique mapped book_ids: 20001
overlap: 19935
overlap rate: 0.5480713716218074
before merge: 1313610
after merge: 909145
coverage: 0.6920965887896712
Saved: /content/data/processed/work_mapped_interactions/poetry_read_work_mapped.parquet
Final shape: (906043, 3)
Processing: romance
wget -c -O "/content/data/interactions/goodreads_interactions_romance.json.gz" "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_romance.json.gz"
--2026-06-04 05:31:34--  https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_romance.json.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2292928849 (2.1G) [application/gzip]
Saving to: ‘/content/data/interactions/goodreads_interactions_romance.json.gz’

/content/data/inter 100%[======

Filtering goodreads_interactions_romance.json.gz: 0it [00:00, ?it/s]

unique interaction book_ids: 331884
unique mapped book_ids: 176803
overlap: 173759
overlap rate: 0.5235534102276699
before merge: 21174642
after merge: 15438615
coverage: 0.7291086668667173
Saved: /content/data/processed/work_mapped_interactions/romance_read_work_mapped.parquet
Final shape: (15380828, 3)
Processing: young
wget -c -O "/content/data/interactions/goodreads_interactions_young_adult.json.gz" "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_young_adult.json.gz"
--2026-06-04 05:48:53--  https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_interactions_young_adult.json.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1840546134 (1.7G) [application/gzip]
Saving to: ‘/content/data/interactions/goodreads_interactions_young_adult.json.gz’

/content

Filtering goodreads_interactions_young_adult.json.gz: 0it [00:00, ?it/s]

unique interaction book_ids: 93046
unique mapped book_ids: 51269
overlap: 51056
overlap rate: 0.5487178384884895
before merge: 15722749
after merge: 13096195
coverage: 0.8329456254755451
Saved: /content/data/processed/work_mapped_interactions/young_read_work_mapped.parquet
Final shape: (13032954, 3)


In [13]:
test = pd.read_parquet('data/processed/work_mapped_interactions/children_read_work_mapped.parquet')

In [14]:
test.head()

,user_id,work_id,rating
0,00000377eea48021d3002730d56aca9a,2402163,5
1,00000377eea48021d3002730d56aca9a,2543234,4
2,00000377eea48021d3002730d56aca9a,2933712,1
3,00000377eea48021d3002730d56aca9a,3183592,3
4,00004584d524ec468619e81b176cc991,1069597,4


In [15]:
test.shape

(5340525, 3)

In [16]:
child = pd.read_csv('data/children.csv')

In [17]:
child.columns

Index(['work_id', 'clean_title', 'author_ids', 'series_list', 'isbn_list',
       'isbn13_list', 'book_id_list', 'publisher_list', 'description',
       'similar_books_list', 'text_reviews_count', 'ratings_count',
       'average_rating', 'num_pages', 'num_pages_missing',
       'popular_shelves_cleaned'],
      dtype='object')